In [ ]:
# merge datasets
from datasets import load_dataset, Dataset
from tqdm import tqdm

# Load both datasets
ds_cot = load_dataset("AI-MO/NuminaMath-CoT", split="train")  # has 'solution'
ds_ppo = load_dataset("RLHFlow/numia_prompt_ppo", split="train")  # no 'solution'

print(ds_cot[0])
print(ds_ppo[0])


In [ ]:
# Step 1: Build a lookup dictionary from the CoT dataset
problem_to_solution = {example["problem"]: example["solution"] for example in ds_cot}

# Step 2: Add 'solution' to PPO dataset by matching on 'problem'
new_examples = []

for example in tqdm(ds_ppo):
    problem = example["problem"]
    solution = problem_to_solution.get(problem, "")  # or None if you prefer
    if solution != "":
        new_example = dict(example)
        new_example["solution"] = solution
        new_examples.append(new_example)

# Step 3: Create new dataset
ds_ppo_with_solution = Dataset.from_list(new_examples)

ds_ppo_with_solution[0]

In [ ]:
len(ds_ppo_with_solution)

In [ ]:
from huggingface_hub import create_repo

#create_repo("numia_prompt_ppo", repo_type="dataset")

ds_ppo_with_solution.push_to_hub("Chenlu123/numia_prompt_ppo")


*Given the gold and solution, verify the math score*

In [ ]:
from math_verify.metric import math_metric
from math_verify.parser import LatexExtractionConfig, ExprExtractionConfig
from math_verify.errors import TimeoutException

In [ ]:
def compute_score(model_output: str, ground_truth: str, timeout_score: float = 0) -> bool:
    verify_func = math_metric(
        gold_extraction_target=(LatexExtractionConfig(),),
        pred_extraction_target=(ExprExtractionConfig(), LatexExtractionConfig()),
    )
    ret_score = 0.

    # Wrap the ground truth in \boxed{} format for verification
    ground_truth_boxed = "\\boxed{" + ground_truth + "}"
    try:
        ret_score, _ = verify_func([ground_truth_boxed], [model_output])
    except Exception as e:
        pass
    except TimeoutException:
        ret_score = timeout_score

    return ret_score

In [ ]:
from datasets import load_dataset, Dataset
from tqdm import tqdm

# Load both datasets
ds_ppo_with_solution = load_dataset("Chenlu123/numia_prompt_ppo", split="train")

In [ ]:
import numpy as np

score = []
for example in ds_ppo_with_solution:
    s = compute_score(example['solution'], example['reward_model']['ground_truth'])
    score.append(s)

mean_score = np.mean(score)

print(mean_score)

In [ ]:
indices_to_keep = [i for i, val in enumerate(score) if val == 1.0]

# Use the .select method to keep only those examples
filtered_ds = ds_ppo_with_solution.select(indices_to_keep)

print(len(filtered_ds))


In [ ]:
filtered_ds.push_to_hub("Chenlu123/numia_prompt_ppo")

# Process numia data to train

In [ ]:
import os
from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files=os.path.expanduser("~/PRM_filter/data/numina_math/train.parquet")
)

dataset['train'][0]

In [ ]:
# Shuffle the 'train' split
dataset['train'] = dataset['train'].shuffle(seed=42)
dataset['train'][0]

In [ ]:
import os
import argparse
from tqdm import tqdm
from datasets import Dataset

short_system_prompt="""
When tackling complex reasoning tasks, you have access to the following actions. Use them as needed to progress through your thought process.

[ASSESS]

[ADVANCE]

[VERIFY]

[SIMPLIFY]

[SYNTHESIZE]

[PIVOT]

[OUTPUT]

You should strictly follow the format below:

[ACTION NAME]

# Your action step 1

# Your action step 2

# Your action step 3

...

Next action: [NEXT ACTION NAME]

"""

task2ability={'Coding':'code',
              'Math':'math'}

dataset = dataset.filter(lambda example: len(example["problem"]) <= 900)


input_dataset = dataset
for split in ['train']:
    output_dataset=[]
    cur_dataset = input_dataset[split]
    for data_entry in tqdm(cur_dataset):
        cur_data = {
            "data_source": data_entry['data_source'],
            "prompt": [
                {
                    "role": "system",
                    "content": short_system_prompt
                },
                {
                    "role": "user",
                    "content": data_entry['problem']
                }],
            "ability": data_entry['ability'],
            "reward_model": {
                "style": "rule",
                "ground_truth": data_entry['reward_model']['ground_truth'],
            },
            "extra_info": {
                'split': 'dummy',
                'index': 0
            }
        }
        output_dataset.append(cur_data)
    output_dataset = Dataset.from_list(output_dataset)
    output_dataset.to_parquet(os.path.join(os.path.expanduser('~/PRM/data'), f'{split}.parquet'))

In [ ]:
output_dataset[0]